# Calibration under shift — results walkthrough

This notebook reads saved manifests and result tables; it never trains a model. Public-data conclusions belong here only after the full seeded grid has completed. The bundled synthetic demo is an engineering check and is deliberately excluded.

In [ ]:
from pathlib import Path
import json
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yaml
from experiments.analyze import validate_complete_grid

ROOT = Path.cwd()
if not (ROOT / "configs").exists() and (ROOT.parent / "configs").exists():
    ROOT = ROOT.parent
RESULTS = ROOT / "results"
FIGURES = RESULTS / "figures"
plt.rcParams.update({"figure.dpi": 120, "axes.grid": True, "grid.alpha": 0.2})
print(f"repository: {ROOT}")

## 1. Prespecified question

The test is temporal in severity space: does a reliability signal cross its prespecified threshold before clean accuracy has fallen by five percentage points? Thresholds live in a versioned YAML file and are applied without substituting missing crossings.

In [ ]:
with (ROOT / "configs" / "analysis_protocol.yaml").open() as handle:
    protocol = yaml.safe_load(handle)
protocol

## 2. Data provenance and split contract

Training, validation, calibration, and test rows are fixed in manifests. Temperature scaling and APS conformal prediction use only the calibration role. Device corruptions are applied only at evaluation time.

In [ ]:
for dataset in ("smids", "hushem", "kromp"):
    path = ROOT / "data" / "splits" / f"{dataset}.csv"
    if path.exists():
        frame = pd.read_csv(path)
        index = ["fold", "split"] if "fold" in frame else ["split"]
        print(f"\n{dataset}: {len(frame):,} manifest rows")
        display(frame.groupby(index).size().rename("rows").to_frame())
    else:
        print(f"\n{dataset}: no manifest (see results/data_audit.md)")

## 3. Load completed result rows

Only `results/metrics.csv` is treated as scientific output. A missing file means the public-data experiment has not run; this notebook remains executable and says so explicitly.

In [ ]:
metrics_path = RESULTS / "metrics.csv"
required = {"dataset", "model", "seed", "fold", "corruption", "severity", "method", "metric", "value"}
if metrics_path.exists():
    metrics = pd.read_csv(metrics_path)
    missing = required - set(metrics)
    if missing:
        raise ValueError(f"metrics file is missing columns: {sorted(missing)}")
    datasets = set(metrics["dataset"].astype(str))
    demo_only = datasets == {"synthetic_demo"}
    if demo_only and os.getenv("CALIBRATION_NOTEBOOK_ALLOW_DEMO") == "1":
        print("CI-only synthetic fixture: exercising result cells; not scientific output.")
    elif "synthetic_demo" in datasets:
        raise ValueError("synthetic demo rows cannot be treated as scientific results")
    else:
        validate_complete_grid(metrics, protocol)
        print("validated complete prespecified public-data grid")
    print(f"{len(metrics):,} tidy rows from {metrics_path}")
else:
    metrics = pd.DataFrame(columns=sorted(required))
    print("No public-data metrics.csv yet; no scientific finding is claimed.")

## 4. Clean baselines

Accuracy and macro-F1 are summarized across independent seeds (or HuSHeM folds). This is a quick integrity check, not the headline shift analysis.

In [ ]:
if metrics.empty:
    print("Skipped: no public-data metrics.")
else:
    clean = metrics.query("corruption == 'clean' and method == 'raw_softmax'")
    baseline = (clean[clean.metric.isin(["accuracy", "macro_f1", "ece", "nll"])]
                .groupby(["dataset", "model", "metric"]).value
                .agg(["mean", "std", "count"]))
    display(baseline)

## 5. Accuracy and calibration over severity

Corruptions are first averaged equally within each seed, then seeds are summarized. This prevents a corruption with more rows from receiving extra weight.

In [ ]:
def device_summary(metric_name, method):
    names = set(protocol["aggregation"]["device_corruptions"])
    selected = metrics[(metrics.corruption.isin(names)) &
                       (metrics.metric == metric_name) &
                       (metrics.method == method)].copy()
    if selected.empty:
        return pd.DataFrame()
    within = (selected.groupby(["dataset", "model", "seed", "fold", "severity"], as_index=False, dropna=False)
              .value.mean())
    return (within.groupby(["dataset", "model", "severity"]).value
            .agg(["mean", "std", "count"]).reset_index())

accuracy = device_summary("accuracy", "raw_softmax")
ece = device_summary("ece", "raw_softmax")
if accuracy.empty:
    print("Skipped: no complete public-data grid.")
else:
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.8))
    for keys, group in accuracy.groupby(["dataset", "model"]):
        axes[0].plot(group.severity, group["mean"], marker="o", label="/".join(keys))
    for keys, group in ece.groupby(["dataset", "model"]):
        axes[1].plot(group.severity, group["mean"], marker="o", label="/".join(keys))
    axes[0].set(xlabel="Severity", ylabel="Accuracy", title="Accuracy")
    axes[1].set(xlabel="Severity", ylabel="ECE", title="Calibration error")
    axes[0].legend(frameon=False)
    plt.tight_layout()

## 6. Temperature transfer

Temperature is optimized once on clean calibration logits. Comparing raw and temperature-scaled ECE under corruption asks whether that clean post-hoc correction transfers; it is not refit on the shifted test set.

In [ ]:
if metrics.empty:
    print("Skipped: no public-data metrics.")
else:
    names = set(protocol["aggregation"]["device_corruptions"])
    comparison = metrics[(metrics.metric == "ece") &
                         (metrics.method.isin(["raw_softmax", "temperature"])) &
                         (metrics.corruption.isin(names))]
    within = (comparison.groupby(
        ["dataset", "model", "method", "seed", "fold", "severity"],
        as_index=False, dropna=False).value.mean())
    display(within.groupby(["dataset", "model", "method", "severity"]).value
            .agg(["mean", "std", "count"]).head(30))

## 7. Selective prediction and conformal sets

Risk at 80% coverage asks whether abstention can retain safer cases. APS reports empirical coverage and mean set size; its nominal guarantee relies on exchangeability and can fail under shift.

In [ ]:
if metrics.empty:
    print("Skipped: no public-data metrics.")
else:
    names = set(protocol["aggregation"]["device_corruptions"])
    selected = metrics[metrics.metric.isin([
        "risk_at_80_coverage", "conformal_coverage", "conformal_mean_set_size"
    ]) & metrics.corruption.isin(names)]
    within = (selected.groupby(
        ["dataset", "model", "method", "metric", "seed", "fold", "severity"],
        as_index=False, dropna=False).value.mean())
    display(within.groupby(
        ["dataset", "model", "method", "metric", "severity"]
    ).value.agg(["mean", "std", "count"]).head(40))

## 8. Prespecified threshold result

The generated table is the direct answer to the thesis. An early-warning gap is positive only when a reliability threshold crosses at a lower severity than the five-point accuracy-loss threshold.

In [ ]:
threshold_path = RESULTS / "thresholds.csv"
if threshold_path.exists():
    thresholds = pd.read_csv(threshold_path)
    display(thresholds)
else:
    print("No threshold table yet. Run: python -m experiments.analyze")

## 9. Attribution stability

Grad-CAM comparisons use fixed clean targets and paired images. Spearman agreement and top-20% saliency IoU quantify drift; pictures alone are not treated as evidence.

In [ ]:
stability_path = FIGURES / "attribution_stability.csv"
if stability_path.exists():
    stability = pd.read_csv(stability_path)
    display(stability.groupby("severity")[["spearman", "top_percent_iou"]]
            .agg(["mean", "std", "count"]))
else:
    print("No attribution run yet. Use experiments/run_attribution.py after training.")

## 10. Interpretation discipline

A complete run should distinguish: (1) classification degradation, (2) miscalibration, (3) uncertainty quality for failure ranking, (4) conformal coverage under broken exchangeability, and (5) input-shift detection. These answer different operational questions and are not interchangeable.

## 11. Limitations

SMIDS and HuSHeM are public proxy datasets with no released donor mapping; Kromp cannot currently be split by patient from its public metadata. Corruptions are sensitivity analyses, not paired smartphone/clinical captures. The study makes no clinical-performance or deployment claim.

## 12. Primary references

- [Thirumalaraju et al., Fertility and Sterility (online 2025; issue 2026)](https://doi.org/10.1016/j.fertnstert.2025.08.021)
- [Kanakasabapathy et al., Nature Biomedical Engineering (2021)](https://doi.org/10.1038/s41551-021-00733-w)
- [Guo et al., ICML (2017)](https://proceedings.mlr.press/v70/guo17a.html)
- [Ovadia et al., NeurIPS (2019)](https://proceedings.neurips.cc/paper_files/paper/2019/hash/8558cb408c1d76621371888657d2eb1d-Abstract.html)
- [Angelopoulos & Bates (2023)](https://doi.org/10.1561/2200000101)

## 13. Next experiment

The decisive extension is a paired acquisition study: image the same specimen through a reference microscope and the lab's smartphone hardware, then compare this synthetic severity ordering with real device-domain metrics while keeping patients grouped and calibration data isolated.